# ChakraModel V6 Strict Verification - True Anti-Fabrication Harness

This notebook securely writes BOTH `verify_kaggle.py` and `harness_core.py` directly to the disk, avoiding the need to upload the Anti-Fabrication Toolkit separately!

**Before running:**
1. Make sure you have added your Image Dataset, Code Dataset, and Weights Dataset.
2. Ensure you run this using **Save Version -> Save & Run All (Commit)**.

In [ ]:
!pip install -q ultralytics timm opencv-python-headless pycocotools

In [ ]:
import os
import sys
import shutil
import glob
import json
import re

###################################################################
# CONFIGURATION
###################################################################

# 1. Update THIS path to the exact subfolder (Make sure images/ and masks/ are directly inside it!)
DATASET_ROOT = '/kaggle/input/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/Center_1'

# 2. Update THIS name so your CSV and Verification Card are labeled correctly
VERIFY_DATASET_NAME = "PolypDB_Center_1"

WORKING_DATASET = '/kaggle/working/dataset_copy'

if os.path.exists(WORKING_DATASET):
    shutil.rmtree(WORKING_DATASET)
shutil.copytree(DATASET_ROOT, WORKING_DATASET)
print(f"Dataset securely copied to {WORKING_DATASET}")

In [ ]:
%%writefile /kaggle/working/harness_core.py
"""
harness_core.py â€” the project-agnostic verification engine.

Nothing in this file knows about ChakraModel or any specific project.
It only knows how to:
  1. plant unpredictable canaries in a dataset folder
  2. run a script live and capture its real stdout with real timestamps
  3. check nonce / canary / count / timing plausibility
  4. sign the verdict so it can't be edited afterward

Used by verifyai.py â€” you should not need to edit this file for a new
project. Only the registry (config/projects.json, managed via
`verifyai add-project`) changes per project.
"""

import os
import re
import cv2
import sys
import json
import time
import hmac
import hashlib
import secrets
import subprocess
import numpy as np


def _dice(a: np.ndarray, b: np.ndarray) -> float:
    a = (a > 0).astype(np.uint8)
    b = (b > 0).astype(np.uint8)
    inter = (a & b).sum()
    denom = a.sum() + b.sum()
    return 1.0 if denom == 0 else float(2 * inter / denom)


def plant_multi_canary(dataset_root: str, images_subdir="images", masks_subdir="masks"):
    n_canaries = secrets.randbelow(3) + 2  # 2..4
    img_dir = os.path.join(dataset_root, images_subdir)
    mask_dir = os.path.join(dataset_root, masks_subdir)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)

    canaries = []
    for _ in range(n_canaries):
        tag = secrets.token_hex(6)
        name = f"CANARY_{tag}.png"
        rng = np.random.default_rng(int(tag, 16))
        img = rng.integers(0, 255, (448, 448, 3), dtype=np.uint8)
        mask = np.zeros((448, 448), dtype=np.uint8)
        side = 60 + secrets.randbelow(80)
        x0 = secrets.randbelow(448 - side)
        y0 = secrets.randbelow(448 - side)
        mask[y0:y0 + side, x0:x0 + side] = 255

        img_path = os.path.join(img_dir, name)
        mask_path = os.path.join(mask_dir, name)
        cv2.imwrite(img_path, img)
        cv2.imwrite(mask_path, mask)

        canaries.append({
            "filename": name,
            "img_md5": hashlib.md5(open(img_path, "rb").read()).hexdigest(),
            "mask_md5": hashlib.md5(open(mask_path, "rb").read()).hexdigest(),
        })

    nonce = secrets.token_hex(8)
    hmac_key = secrets.token_bytes(32)

    return {
        "nonce": nonce,
        "hmac_key": hmac_key,
        "canaries": canaries,
        "planted_at_epoch": time.time(),
    }


def run_live(script_path: str, extra_env: dict, timeout_sec: int = 3600):
    env = os.environ.copy()
    env.update(extra_env)

    start = time.monotonic()
    proc = subprocess.Popen(
        [sys.executable, script_path],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        bufsize=1,
    )

    lines, arrivals = [], []
    while True:
        line = proc.stdout.readline()
        if not line and proc.poll() is not None:
            break
        if line:
            lines.append(line)
            arrivals.append(time.monotonic() - start)
        if time.monotonic() - start > timeout_sec:
            proc.kill()
            lines.append("\n[HARNESS] TIMEOUT - process killed.\n")
            break

    proc.wait()
    elapsed = time.monotonic() - start
    return "".join(lines), arrivals, elapsed, proc.returncode


def extract_section(full_text: str, section_label: str) -> str:
    if not section_label:
        return full_text
    marker = f"VERIFYING DATASET: {section_label}"
    start = full_text.find(marker)
    if start == -1:
        return ""
    rest = full_text[start:]
    next_pos = rest.find("VERIFYING DATASET:", len(marker))
    return rest if next_pos == -1 else rest[:next_pos]


def verify(full_text: str, arrivals: list, elapsed_sec: float,
           secret: dict, claimed_total_images: int,
           real_total_images_on_disk: int,
           images_per_sec_ceiling: float = 15.0,
           min_plausible_sec: float = 2.0):
    findings = []
    passed = True

    for c in secret["canaries"]:
        name = c["filename"]
        if name not in full_text:
            passed = False
            findings.append(f"FAIL: canary {name} never appears in live output.")
            continue
        m = re.search(rf"{re.escape(name)}.*?([01]\.\d+)", full_text)
        if not m:
            passed = False
            findings.append(f"FAIL: canary {name} mentioned but no dice value found next to it.")
            continue
        val = float(m.group(1))
        findings.append(f"INFO: canary {name} reported dice = {val:.4f}")
        if val > 0.35:
            passed = False
            findings.append(f"FAIL: canary {name} score implausibly high for pure noise input.")

    if secret["nonce"] not in full_text:
        passed = False
        findings.append("FAIL: session nonce absent from live-captured stdout.")
    else:
        findings.append("PASS: nonce present in harness-captured stdout.")

    if claimed_total_images != real_total_images_on_disk:
        passed = False
        findings.append(
            f"FAIL: claimed count ({claimed_total_images}) != actual disk count "
            f"({real_total_images_on_disk})."
        )

    min_required_sec = max(min_plausible_sec, claimed_total_images / images_per_sec_ceiling)
    if elapsed_sec < min_required_sec:
        passed = False
        findings.append(
            f"FAIL: wall-clock elapsed ({elapsed_sec:.2f}s) faster than plausible "
            f"for {claimed_total_images} images (needs >= {min_required_sec:.2f}s)."
        )
    else:
        findings.append(f"PASS: elapsed time ({elapsed_sec:.2f}s) is plausible.")

    if len(arrivals) > 5:
        spread = arrivals[-1] - arrivals[0]
        if spread < 1.0 and claimed_total_images > 20:
            passed = False
            findings.append(
                f"FAIL: all {len(arrivals)} output lines arrived within {spread:.3f}s "
                f"of each other â€” real per-image logging should be spread out."
            )

    return {"pass": passed, "findings": findings, "elapsed_sec": elapsed_sec,
            "n_output_lines": len(arrivals)}


def sign_verdict(verdict: dict, hmac_key: bytes) -> dict:
    payload = json.dumps(verdict, sort_keys=True).encode()
    sig = hmac.new(hmac_key, payload, hashlib.sha256).hexdigest()
    return {"verdict": verdict, "signature": sig}


def check_signature(signed_blob: dict, hmac_key: bytes) -> bool:
    payload = json.dumps(signed_blob["verdict"], sort_keys=True).encode()
    expected = hmac.new(hmac_key, payload, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, signed_blob["signature"])


def write_completion_card(verdict_signed: dict, claim: dict, project: str,
                           verdicts_root: str):
    out_dir = os.path.join(verdicts_root, project, claim.get("dataset", "unknown"))
    os.makedirs(out_dir, exist_ok=True)
    decision = "ADMITTED" if verdict_signed["verdict"]["pass"] else "WITHHELD"
    card = {
        "decision": decision,
        "claim": claim,
        "verdict": verdict_signed["verdict"],
        "signature": verdict_signed["signature"],
        "written_at_epoch": time.time(),
    }
    fname = f"{claim.get('metric_name','metric')}_{int(time.time())}.json"
    path = os.path.join(out_dir, fname)
    with open(path, "w") as f:
        json.dump(card, f, indent=2)
    return path, decision



In [ ]:
%%writefile /kaggle/working/verify_kaggle.py
import os
import sys
import glob
import cv2
import numpy as np
import torch
from pathlib import Path
import hashlib

def md5(fname):
    hash_md5 = hashlib.md5()
    with open(fname, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

def verify_strict():
    print("=" * 60)
    print("STRICT CHECKPOINT AND ARCHITECTURE VERIFICATION")
    print("=" * 60)
    
    # Identify codebase path
    code_dir = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'chakranet_segmenter.py' in files and 'src' in root:
            code_dir = root
            break
    if not code_dir:
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'chakranet_segmenter.py' in files:
                code_dir = root
                break
    assert code_dir, "FATAL: ChakraModel codebase not found in /kaggle/input"
    sys.path.insert(0, code_dir)
    print(f"[VERIFY] Codebase found at: {code_dir}")
    
    from ultralytics import YOLO
    from chakranet_segmenter import ChakraNet
    from metrics_engine_v2 import MetricsEngineV2

    # BUG 3 FIX: Strict Weight Selection
    yolo_candidates = glob.glob('/kaggle/input/**/best.pt', recursive=True)
    vit_candidates = glob.glob('/kaggle/input/**/chakra_transformer_best.pth', recursive=True)
    
    # Strictly filter yolo_candidates (ignore anything with 'transformer' in the path)
    yolo_candidates = [p for p in yolo_candidates if 'transformer' not in p.lower()]
    
    print(f"YOLO candidates found: {yolo_candidates}")
    print(f"ViT candidates found: {vit_candidates}")
    
    assert len(yolo_candidates) == 1, f"FATAL: Expected exactly 1 YOLO weight best.pt, found {len(yolo_candidates)}!"
    assert len(vit_candidates) == 1, f"FATAL: Expected exactly 1 ViT weight chakra_transformer_best.pth, found {len(vit_candidates)}!"
    
    yolo_path = yolo_candidates[0]
    weight_path = vit_candidates[0]

    print(f"Segmenter Checkpoint: {weight_path}")
    print(f"Segmenter MD5: {md5(weight_path)}")
    print(f"YOLO Checkpoint: {yolo_path}")
    print(f"YOLO MD5: {md5(yolo_path)}")
    print("=" * 60)
    
    nonce = os.environ.get("VERIFY_NONCE", "NO_NONCE")
    print(f"VERIFY_NONCE: {nonce}")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Executing on device: {device}")
    
    yolo_model = YOLO(yolo_path)
    yolo_model.to(device)
    
    segmenter = ChakraNet(img_size=(384, 384), device=device)
    segmenter.model.to(device)
    
    # Load weights correctly by explicitly stripping 'module.'
    sd = torch.load(weight_path, map_location=device)
    sd_fixed = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}
    res = segmenter.model.load_state_dict(sd_fixed, strict=False)
    print(f"ChakraNet Weights Loaded. Missing keys: {len(res.missing_keys)}, Unexpected keys: {len(res.unexpected_keys)}")
    
    total_keys = len(segmenter.model.state_dict())
    if len(res.missing_keys) > (0.1 * total_keys):
        print(f"[FATAL] Major weight mismatch! Missing keys > 10% ({len(res.missing_keys)} missing out of {total_keys}).")
        sys.exit(1)
    
    metrics_engine = MetricsEngineV2()
    
    dataset_name = os.environ.get("VERIFY_DATASET_NAME")
    dataset_root = os.environ.get("VERIFY_DATASET_ROOT")
    
    if not dataset_name or not dataset_root:
        print("[FATAL] VERIFY_DATASET_NAME or VERIFY_DATASET_ROOT environment variable not set.")
        sys.exit(1)
        
    print(f"\n============================================================")
    print(f"VERIFYING DATASET: {dataset_name}")
    print(f"============================================================")
    
    d_root = Path(dataset_root)
    images_dir = d_root / "images"
    masks_dir = d_root / "masks"
    
    if not images_dir.exists():
        print(f"[FATAL] Images directory not found: {images_dir}")
        sys.exit(1)
        
    image_paths = sorted(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.tif")))
    print(f"Total images found: {len(image_paths)}")
    
    results = {k: [] for k in ["Dice", "mIoU", "wF-measure", "S-measure", "E-measure", "MAE"]}
    import csv
    csv_file = "/kaggle/working/per_image_metrics_log.csv"
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Filename", "Dice", "mIoU", "wF-measure", "S-measure", "E-measure", "MAE"])
        
    for i, img_path in enumerate(image_paths):
        mask_path = None
        for ext in [".png", ".jpg", ".tif", ".bmp"]:
            potential_path = masks_dir / (img_path.stem + ext)
            if potential_path.exists():
                mask_path = potential_path
                break
                
        if not mask_path:
            print(f"[WARN] Mask not found for {img_path.name}")
            continue
            
        img_bgr = cv2.imread(str(img_path))
        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        gt_resized = cv2.resize(gt_mask, (448, 448), interpolation=cv2.INTER_NEAREST)
        
        img_resized = cv2.resize(img_bgr, (448, 448))
        yolo_results = yolo_model(img_resized, verbose=False)[0]
        
        # Soft prob map initialized to 0
        c_prob = np.zeros((448, 448), dtype=np.float32)
        
        if len(yolo_results.boxes) > 0:
            boxes = sorted(yolo_results.boxes, key=lambda b: b.conf[0].item(), reverse=True)
            box = boxes[0].xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, box)
            
            w_box, h_box = x2 - x1, y2 - y1
            px, py = int(0.25 * w_box), int(0.25 * h_box)
            px1, py1 = max(0, x1 - px), max(0, y1 - py)
            px2, py2 = max(0, min(448, x2 + px)), max(0, min(448, y2 + py))
            
            if px2 > px1 and py2 > py1:
                roi_crop = img_resized[py1:py2, px1:px2]
                
                # BUG 2 FIX: Direct manual inference EXACTLY matching chakranet_segmenter.py
                from utils.transforms import letterbox_pad, unletterbox
                img_rgb = cv2.cvtColor(roi_crop, cv2.COLOR_BGR2RGB)
                img_resized_crop, meta = letterbox_pad(img_rgb, target_size=(384, 384))
                
                img_norm = img_resized_crop.astype(np.float32) / 255.0
                mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
                std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
                img_norm = (img_norm - mean) / std
                img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).to(device)
                
                try:
                    with torch.no_grad():
                        logits = segmenter.model(img_tensor)
                        prob = torch.sigmoid(logits)
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        print("[WARN] Switched to CPU due to CUDA OutOfMemory!")
                        img_tensor_cpu = img_tensor.cpu()
                        segmenter.model.to('cpu')
                        with torch.no_grad():
                            logits = segmenter.model(img_tensor_cpu)
                            prob = torch.sigmoid(logits)
                        segmenter.model.to(device)
                    else:
                        raise
                        
                prob_np = prob.squeeze().cpu().numpy()
                prob_resized = unletterbox(prob_np, meta)
                
                c_prob[py1:py2, px1:px2] = prob_resized
        
        # Explicit binarization ONLY for the ground truth here (network outputs soft prob)
        gt_bin = (gt_resized > 127).astype(np.uint8)
        
        # Pass continuous probability map (c_prob) directly to compute_all 
        # (compute_all handles thresholding for Dice/IoU internally)
        metrics = metrics_engine.compute_all(c_prob, gt_bin)
        
        for k in results.keys():
            results[k].append(metrics[k])
            
        # BUG 4 FIX: Checkpoint saving per image
        with open(csv_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([img_path.name, metrics['Dice'], metrics['mIoU'], metrics['wF-measure'], metrics['S-measure'], metrics['E-measure'], metrics['MAE']])
            
        # IMPORTANT: Print per-image dice WITH FILENAME for canary detection
        dice_score = metrics['Dice']
        print(f"File: {img_path.name} | Dice: {dice_score:.4f}")
        
    print(f"\n[DONE] {dataset_name}")
    print(f"Total Evaluated Images: {len(results['Dice'])}")
    for k in results.keys():
        print(f"Final True Average {k}: {float(np.mean(results[k])):.4f}")
    print(f"============================================================")

if __name__ == "__main__":
    verify_strict()



In [ ]:
sys.path.insert(0, '/kaggle/working')

from harness_core import plant_multi_canary, run_live, verify, sign_verdict

print("Planting canaries...")
secret = plant_multi_canary(WORKING_DATASET)

extra_env = {
    "VERIFY_NONCE": secret["nonce"],
    "VERIFY_DATASET_NAME": VERIFY_DATASET_NAME,
    "VERIFY_DATASET_ROOT": WORKING_DATASET
}

print(f"\nRunning verification for {VERIFY_DATASET_NAME}...")
stdout_text, arrivals, elapsed, retcode = run_live('/kaggle/working/verify_kaggle.py', extra_env, timeout_sec=10000)

print(stdout_text)

if retcode != 0:
    print(f"FATAL: verify_kaggle.py exited with code {retcode}")
else:
    # Parse claimed total from stdout
    match = re.search(r"Total Evaluated Images:\s*(\d+)", stdout_text)
    claimed_total = int(match.group(1)) if match else -1
    
    # Use harness core to verify output logic
    real_count = len(os.listdir(os.path.join(WORKING_DATASET, "images")))
    verdict_dict = verify(stdout_text, arrivals, elapsed, secret, claimed_total, real_count)
    
    card = {
        "project": "ChakraModel",
        "dataset": VERIFY_DATASET_NAME,
        "nonce": secret["nonce"],
        "decision": "ADMITTED" if verdict_dict["pass"] else "WITHHELD",
        "findings": verdict_dict["findings"],
        "elapsed_sec": verdict_dict["elapsed_sec"]
    }
    
    signed_card = sign_verdict(card, secret["hmac_key"])
    
    with open('/kaggle/working/completion_card.json', 'w') as f:
        json.dump(signed_card, f, indent=2)
        
    print(f"\n=========================================")
    print(f"VERDICT: {card['decision']}")
    for finding in card['findings']:
        print(f"  - {finding}")
    print(f"Saved to completion_card.json")
    print(f"=========================================")
